# PUDM — Flow Matching Training & Evaluation

This notebook trains and evaluates **PUDM** using the **Flow Matching** strategy (Conditional Flow Matching with linear interpolation and Euler ODE sampling).

For the DDPM baseline, see `pudm_ddpm.ipynb`.

**Prerequisites:** Run `config.ipynb` once to clone the repo to Drive and compile CUDA extensions.

## 1. Setup

In [ ]:
import os, sys, shutil, glob as _glob

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Repo lives on Drive (cloned by config.ipynb)
REPO_DIR = '/content/drive/MyDrive/MVA/pointcloud/pudm_extension'
assert os.path.isdir(REPO_DIR), f'Repo not found at {REPO_DIR} — run config.ipynb first'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

# Install Python dependencies
!pip install -q -r requirements.txt

import torch
print(f'PyTorch: {torch.__version__}  CUDA: {torch.version.cuda}')
print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Restore cached CUDA extensions (compiled by config.ipynb)
_tag = f'py{sys.version_info.major}{sys.version_info.minor}_pt{torch.__version__.split("+")[0]}_cu{torch.version.cuda.replace(".", "")}'
CACHE_DIR = os.path.join(REPO_DIR, 'cache', _tag)

POINTOPS_SRC = os.path.join(REPO_DIR, 'src', 'ops', 'pointops')
PNET2_PKG    = os.path.join(REPO_DIR, 'src', 'ops', 'pointnet2_ops')

# pointops
for so in _glob.glob(os.path.join(CACHE_DIR, 'pointops', 'pointops_cuda*.so')):
    shutil.copy2(so, POINTOPS_SRC)
    print(f'✓ pointops: {os.path.basename(so)}')

# pointnet2_ops
for so in _glob.glob(os.path.join(CACHE_DIR, 'pointnet2_ops', '_ext*.so')):
    shutil.copy2(so, PNET2_PKG)
    print(f'✓ pointnet2_ops: {os.path.basename(so)}')

if not _glob.glob(os.path.join(CACHE_DIR, '**', '*.so'), recursive=True):
    raise FileNotFoundError(f'No cached .so files in {CACHE_DIR} — run config.ipynb first')

# Verify imports
from src.generative import get_strategy, STRATEGIES
from src.utils.config import load_config
from src.utils.misc import set_seed

print(f'\nStrategies: {list(STRATEGIES.keys())}')
print('All imports OK!')

## 2. Configuration

Choose your dataset and data paths.

In [ ]:
# ============ EDIT THESE ============
DATASET = 'PU1K'                    # 'PU1K' or 'PUGAN'
STRATEGY_NAME = 'flow_matching'     # fixed for this notebook
R = 4                               # upsampling rate
GAMMA = 0.5                         # condition interpolation
SEED = 42
# ====================================

import shutil, time as _time
from src.utils.config import get_strategy_config

# Copy dataset from Drive to local SSD for fast I/O
DATA_DRIVE = os.path.join(REPO_DIR, 'data', DATASET)
DATA_DIR   = os.path.join('/content/data', DATASET)

if os.path.isdir(DATA_DIR):
    print(f'✓ {DATASET}: already on local disk')
elif os.path.isdir(DATA_DRIVE):
    t0 = _time.time()
    print(f'{DATASET}: copying from Drive to local SSD ...')
    shutil.copytree(DATA_DRIVE, DATA_DIR)
    print(f'✓ {DATASET}: copied in {_time.time()-t0:.1f}s')
else:
    raise FileNotFoundError(
        f'Dataset not found at {DATA_DRIVE}\n'
        f'Place your {DATASET} data in {DATA_DRIVE} (see config.ipynb).'
    )

CHECKPOINT_DIR = os.path.join(REPO_DIR, 'checkpoints', DATASET.lower(), STRATEGY_NAME)
OUTPUT_DIR = os.path.join(REPO_DIR, 'outputs', DATASET.lower())

set_seed(SEED)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load config
config_path = os.path.join(REPO_DIR, 'configs', f'{DATASET}.json')
config = load_config(config_path)

# Override data directory in config (point at local copy)
dataset_key = 'pu1k_dataset_config' if DATASET == 'PU1K' else 'pugan_dataset_config'
config[dataset_key]['data_dir'] = DATA_DIR

# Get strategy and its config section
strategy = get_strategy(STRATEGY_NAME)
strategy_config = get_strategy_config(config, STRATEGY_NAME)
hyperparams = strategy.compute_hyperparams(**strategy_config)

# Move hyperparams to GPU
for key in hyperparams:
    if key != 'T' and isinstance(hyperparams[key], torch.Tensor):
        hyperparams[key] = hyperparams[key].cuda()

print(f'Dataset: {DATASET}')
print(f'Strategy: {strategy.name}')
print(f'Strategy config: {strategy_config}')
print(f'Data dir: {DATA_DIR} (local SSD)')
print(f'Checkpoints: {CHECKPOINT_DIR}')
print(f'T = {hyperparams["T"]}')

## 3. Training

Train the model with Flow Matching. Skip this cell if you already have a checkpoint.

In [ ]:
import time
import torch.nn as nn
from torch.amp import GradScaler, autocast
from src.data.dataset import get_dataloader
from src.models.pointnet2_with_pcld_condition import PointNet2CloudCondition

# Confirm actual GPU right before training (device name can be stale)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
print()

# Training params
train_config = config['train_config']
pointnet_config = config['pointnet_config']
trainset_config = config[dataset_key]

# Auto-scale batch size & workers for the current GPU
gpu_name = torch.cuda.get_device_name(0).lower()
if 'a100' in gpu_name or 'h100' in gpu_name or 'a10' in gpu_name:
    trainset_config['batch_size'] = 32
    trainset_config['num_workers'] = 8
    print(f'High-end GPU detected → batch_size=32, num_workers=8')
elif 'v100' in gpu_name or 'l4' in gpu_name:
    trainset_config['batch_size'] = 32
    trainset_config['num_workers'] = 4
    print(f'Mid-tier GPU detected → batch_size=32, num_workers=4')
else:
    print(f'Using config defaults → batch_size={trainset_config["batch_size"]}, num_workers={trainset_config["num_workers"]}')

N_EPOCHS = 200                      # original paper uses 2000; 200 is enough for comparison
LR = train_config.get('learning_rate', 2e-4)
EPOCHS_PER_CKPT = train_config.get('epochs_per_ckpt', 10)
LOG_EVERY = train_config.get('iters_per_logging', 50)

# DataLoader
trainloader = get_dataloader(trainset_config)
print(f'Training samples: {len(trainloader.dataset)}')
print(f'Batches per epoch: {len(trainloader)}')

# Model
net = PointNet2CloudCondition(pointnet_config).cuda()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
loss_fn = nn.MSELoss()
scaler = GradScaler('cuda')

# Resume from checkpoint if available
start_epoch = 0
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')]) if os.path.exists(CHECKPOINT_DIR) else []
if ckpt_files:
    latest = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
    ckpt = torch.load(latest, map_location='cpu')
    net.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt.get('epoch', 0)
    print(f'Resumed from {latest} at epoch {start_epoch}')

print(f'\nTraining {strategy.name} for {N_EPOCHS} epochs (starting from {start_epoch})...')

In [ ]:
# Loss tracking buffers
iter_losses  = []   # list of (n_iter, loss) logged every LOG_EVERY iters
epoch_losses = []   # list of (epoch, avg_loss) per epoch

# Restore loss history if resuming
if ckpt_files and 'iter_losses' in ckpt:
    iter_losses  = ckpt.get('iter_losses', [])
    epoch_losses = ckpt.get('epoch_losses', [])
    print(f'Restored loss history: {len(epoch_losses)} epochs, {len(iter_losses)} log points')

# Training loop (mixed precision)
net.train()
n_iter = start_epoch * len(trainloader)
n_iters_total = N_EPOCHS * len(trainloader)
time0 = time.time()

for epoch in range(start_epoch + 1, N_EPOCHS + 1):
    epoch_loss = 0.0
    for data in trainloader:
        label     = data['label'].cuda()
        X         = data['complete'].cuda()
        condition = data['partial'].cuda()

        optimizer.zero_grad()
        with autocast('cuda'):
            loss = strategy.training_loss(net, loss_fn, X, hyperparams,
                                          label=label, condition=condition)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        n_iter += 1

        if n_iter % LOG_EVERY == 0:
            iter_losses.append((n_iter, loss.item()))
            print(f'  epoch {epoch}/{N_EPOCHS}  iter {n_iter}/{n_iters_total}  loss: {loss.item():.6f}')

    avg_loss = epoch_loss / len(trainloader)
    epoch_losses.append((epoch, avg_loss))

    if epoch % EPOCHS_PER_CKPT == 0:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f'pointnet_ckpt_{epoch}_{avg_loss:.6f}.pkl')
        torch.save({
            'epoch': epoch,
            'iter': n_iter,
            'model_state_dict': net.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'training_time_seconds': int(time.time() - time0),
            'strategy': strategy.name,
            'iter_losses': iter_losses,
            'epoch_losses': epoch_losses,
        }, ckpt_path)
        print(f'  -> Saved checkpoint: {ckpt_path}')

print(f'\nTraining complete. Total time: {(time.time() - time0)/60:.1f} min')

## 4. Loss Curves

Plot per-epoch and per-iteration training loss. Run this cell at any point after (or during) training.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def ema(values, alpha=0.05):
    s, out = values[0], [values[0]]
    for v in values[1:]:
        s = alpha * v + (1 - alpha) * s
        out.append(s)
    return np.array(out)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle(f'{strategy.name} Training Loss — {DATASET}', fontsize=13)

if epoch_losses:
    ep_x, ep_y = zip(*epoch_losses)
    axes[0].plot(ep_x, ep_y, color='darkorange', linewidth=1.5)
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Avg MSE Loss')
    axes[0].set_title('Per-epoch average loss')
    axes[0].set_yscale('log'); axes[0].grid(True, alpha=0.3)
else:
    axes[0].text(0.5, 0.5, 'No epoch data yet', ha='center', va='center', transform=axes[0].transAxes)

if iter_losses:
    it_x, it_y = zip(*iter_losses)
    it_x, it_y = np.array(it_x), np.array(it_y)
    axes[1].plot(it_x, it_y, color='moccasin', linewidth=0.5, alpha=0.4, label='raw')
    axes[1].plot(it_x, ema(it_y), color='darkorange', linewidth=1.5, label='EMA (α=0.05)')
    axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('MSE Loss')
    axes[1].set_title('Per-iteration loss (EMA smoothed)')
    axes[1].set_yscale('log'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, 'No iteration data yet', ha='center', va='center', transform=axes[1].transAxes)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, f'training_loss_{strategy.name.lower()}.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

## 5. Sampling & Evaluation

Generate dense point clouds from the test set and compute metrics.

In [ ]:
from src.scripts.eval import evaluate

# Testing always uses PUGAN dataset (PU1K has no test split)
TEST_DATASET = 'PUGAN'
TEST_DATA_DRIVE = os.path.join(REPO_DIR, 'data', TEST_DATASET)
TEST_DATA_DIR   = os.path.join('/content/data', TEST_DATASET)

if os.path.isdir(TEST_DATA_DIR):
    print(f'✓ {TEST_DATASET} test data: already on local disk')
elif os.path.isdir(TEST_DATA_DRIVE):
    t0 = _time.time()
    print(f'{TEST_DATASET}: copying from Drive to local SSD ...')
    shutil.copytree(TEST_DATA_DRIVE, TEST_DATA_DIR)
    print(f'✓ {TEST_DATASET}: copied in {_time.time()-t0:.1f}s')
else:
    raise FileNotFoundError(
        f'Test dataset not found at {TEST_DATA_DRIVE}\n'
        f'Place PUGAN data (with test/ folder) in {TEST_DATA_DRIVE}.'
    )

# Choose checkpoint
ckpt_files = sorted([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith('.pkl')])
if ckpt_files:
    CKPT_PATH = os.path.join(CHECKPOINT_DIR, ckpt_files[-1])
else:
    raise FileNotFoundError(f'No checkpoints in {CHECKPOINT_DIR}')

print(f'Using checkpoint: {CKPT_PATH}')

# Load model
net_eval = PointNet2CloudCondition(pointnet_config).cuda()
ckpt = torch.load(CKPT_PATH, map_location='cpu')
net_eval.load_state_dict(ckpt['model_state_dict'], strict=False)
net_eval.eval()

# Test dataloader — use PUGAN config for testing
pugan_config_path = os.path.join(REPO_DIR, 'configs', 'PUGAN.json')
pugan_config = load_config(pugan_config_path)
test_config = pugan_config['pugan_dataset_config'].copy()
test_config['data_dir'] = TEST_DATA_DIR
test_config['batch_size'] = 14
test_config['eval_batch_size'] = 14
testloader = get_dataloader(test_config, phase='test')
print(f'Test samples: {len(testloader.dataset)}')

STEP = 30  # Euler integration steps for Flow Matching
SAVE_DIR = os.path.join(OUTPUT_DIR, 'samples')
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
with torch.no_grad():
    CD, HD, P2F, meta, metrics = evaluate(
        net=net_eval,
        testloader=testloader,
        strategy=strategy,
        hyperparams=hyperparams,
        print_every_n_steps=strategy_config['T'] // 5,
        scale=test_config.get('scale', 1),
        compute_cd=True,
        return_all_metrics=True,
        R=R,
        npoints=test_config['npoints'],
        T=strategy_config['T'],
        step=STEP,
        save_dir=SAVE_DIR,
        gamma=GAMMA,
    )

print(f'\n{"="*50}')
print(f'Strategy: {strategy.name}')
print(f'Train dataset: {DATASET} | Test dataset: {TEST_DATASET}')
print(f'Upsampling: {test_config["npoints"]} -> {test_config["npoints"] * R}')
print(f'Chamfer Distance (CD): {CD:.8f}')
print(f'Hausdorff Distance (HD): {HD:.8f}')
print(f'Point-to-Face (P2F): {P2F:.8f}')
print(f'{"="*50}')

## 6. Results Visualization

**Cell 6a** — Per-sample metric distributions: CD histogram, HD histogram, CD-vs-HD scatter.
**Cell 6b** — Qualitative 3-D scatter: sparse input / generated / ground-truth for N_VIS samples.

Requires `metrics` dict from the evaluation cell above.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── 1. Metric distributions ───────────────────────────────────────────────────
cd_vals = metrics['cd_distance'].cpu().numpy()
hd_vals = metrics['h_distance'].cpu().numpy().max(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{strategy.name} — {TEST_DATASET} Test Set Metrics  (n={len(cd_vals)})', fontsize=13)

axes[0].hist(cd_vals, bins=40, color='darkorange', edgecolor='white', linewidth=0.4)
axes[0].axvline(cd_vals.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'mean = {cd_vals.mean():.2e}')
axes[0].axvline(np.median(cd_vals), color='gold', linestyle=':', linewidth=1.5,
                label=f'median = {np.median(cd_vals):.2e}')
axes[0].set_xlabel('Chamfer Distance'); axes[0].set_ylabel('Count')
axes[0].set_title('CD distribution'); axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)

axes[1].hist(hd_vals, bins=40, color='coral', edgecolor='white', linewidth=0.4)
axes[1].axvline(hd_vals.mean(), color='darkred', linestyle='--', linewidth=1.5,
                label=f'mean = {hd_vals.mean():.2e}')
axes[1].axvline(np.median(hd_vals), color='gold', linestyle=':', linewidth=1.5,
                label=f'median = {np.median(hd_vals):.2e}')
axes[1].set_xlabel('Hausdorff Distance'); axes[1].set_ylabel('Count')
axes[1].set_title('HD distribution'); axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)

axes[2].scatter(cd_vals, hd_vals, s=8, alpha=0.5, color='mediumpurple')
axes[2].set_xlabel('Chamfer Distance'); axes[2].set_ylabel('Hausdorff Distance')
axes[2].set_title('CD vs HD per sample'); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, f'metric_distributions_{strategy.name.lower()}.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')
print(f'\nSummary:  CD mean={cd_vals.mean():.6f}  std={cd_vals.std():.6f}  |  '
      f'HD mean={hd_vals.mean():.6f}  std={hd_vals.std():.6f}')

In [ ]:
import numpy as np

# ── 2. Qualitative: input / generated / ground-truth side-by-side ─────────────
N_VIS = 3

vis_batch     = next(iter(testloader))
vis_label     = vis_batch['label'][:N_VIS].cuda()
vis_condition = vis_batch['partial'][:N_VIS].cuda()
vis_gt        = vis_batch['complete'][:N_VIS].cuda()

net_eval.reset_cond_features()
with torch.no_grad():
    vis_gen, _, _ = strategy.sample(
        net=net_eval,
        size=(N_VIS, vis_gt.shape[1], 3),
        hyperparams=hyperparams,
        label=vis_label,
        condition=vis_condition,
        R=R, gamma=GAMMA, num_steps=STEP,
    )

cond_np = vis_condition.cpu().numpy()
gen_np  = vis_gen.cpu().numpy()
gt_np   = vis_gt.cpu().numpy()

def subsample(pts, n=512):
    idx = np.random.choice(len(pts), min(n, len(pts)), replace=False)
    return pts[idx]

rows, cols = 3, N_VIS
fig = plt.figure(figsize=(4 * cols, 4 * rows))
fig.suptitle(
    f'{strategy.name}  |  {vis_condition.shape[1]} pts → {vis_gt.shape[1]} pts  ({STEP} Euler steps)',
    fontsize=12
)
row_labels = ['Input (sparse)', 'Generated (dense)', 'Ground truth (dense)']
colors     = ['steelblue', 'darkorange', 'seagreen']
data_rows  = [cond_np, gen_np, gt_np]

for row, (label_str, color, data) in enumerate(zip(row_labels, colors, data_rows)):
    for col in range(cols):
        ax = fig.add_subplot(rows, cols, row * cols + col + 1, projection='3d')
        pts = subsample(data[col])
        ax.scatter(pts[:, 0], pts[:, 1], pts[:, 2], s=1.5, c=color, alpha=0.7)
        if col == 0:
            ax.set_ylabel(label_str, fontsize=9, labelpad=10)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_zticks([])
        ax.set_box_aspect([1, 1, 1])

plt.tight_layout()
out_path = os.path.join(OUTPUT_DIR, f'qualitative_{strategy.name.lower()}.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out_path}')

## 7. Single-File Example

Upsample a single `.xyz` point cloud and visualize the result.

In [ ]:
import numpy as np
import open3d
from src.utils.pc_utils import pc_normalize, numpy_to_pc

# Path to a single .xyz file
EXAMPLE_FILE = '/content/drive/MyDrive/MVA/pointcloud/data/example/pig.xyz'  # EDIT THIS

if os.path.exists(EXAMPLE_FILE):
    pc = open3d.io.read_point_cloud(EXAMPLE_FILE)
    pts = np.asarray(pc.points, dtype=np.float32)
    pts = pc_normalize(pts)
    condition = torch.from_numpy(pts).unsqueeze(0).cuda()
    npoints = condition.shape[1]
    label = torch.ones(1, dtype=torch.long).cuda() * (R - 1)

    net_eval.eval()
    net_eval.reset_cond_features()

    with torch.no_grad():
        gen, cond_pre, z = strategy.sample(
            net=net_eval, size=(1, npoints * R, 3),
            hyperparams=hyperparams, label=label,
            condition=condition, R=R, gamma=GAMMA, num_steps=STEP,
        )

    gen_np = gen[0].cpu().numpy()
    print(f'Input: {npoints} points -> Generated: {gen_np.shape[0]} points')

    # Save result
    out_path = os.path.join(OUTPUT_DIR, 'example_output.xyz')
    open3d.io.write_point_cloud(out_path, numpy_to_pc(gen_np))
    print(f'Saved to: {out_path}')
else:
    print(f'Example file not found: {EXAMPLE_FILE}')
    print('Edit EXAMPLE_FILE path above, or skip this cell.')